## Verificar os dados

In [1]:
%pip install -r requirements.txt

from pathlib import Path

# Este notebook usa o ficheiro limpo que o 01_EDA grava no fim
LIMPO = Path("data/kingcounty/kc_clean.csv")
if not LIMPO.exists():
    raise FileNotFoundError(f"{LIMPO} não existe. Executa primeiro o notebook 01_EDA.")


print("Dataset limpo em:", LIMPO)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dataset limpo em: data/kingcounty/kc_clean.csv


## Objetivo

Ver que variáveis fazem uma venda ficar barata, normal ou cara em relação ao preço esperado. Usa o `kc_clean.csv` gerado pelo `01_EDA`.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

from libs.decisiontrees import DecisionTreeClassifier

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 250)

SEMENTE = 42
CLASSES = ["baixo", "normal", "alto"]

O preço esperado de cada venda foi estimado com uma regressão Ridge sobre o log do preço, com `sqft_living`, `sqft_lot`, `bedrooms`, `bathrooms`, o código postal e o mês da venda. As duas áreas entram como curvas e as restantes como categorias, em que cada código postal fica com o seu nível de preço. Mais de 12% abaixo do esperado é `baixo`, mais de 12% acima é `alto` e o resto é `normal`.

## Preparar os dados

Todas as vendas têm classe.

In [3]:
dados = pd.read_csv("data/kingcounty/kc_clean.csv", parse_dates=["date"])

vendas = dados[dados["ClassePreco"].notna()].reset_index(drop=True)
print(f"{len(vendas)} vendas com classe (de {len(dados)} linhas)")
vendas["ClassePreco"].value_counts(normalize=True).round(3)

21613 vendas com classe (de 21613 linhas)


ClassePreco
normal    0.529
alto      0.247
baixo     0.224
Name: proportion, dtype: float64

Além das variáveis da regressão entram `floors`, `waterfront`, `view`, `condition`, `grade`, `sqft_basement`, `yr_built`, `sqft_living15`, `sqft_lot15` e os `Meses` desde a primeira venda.

Removemos `price`, `PrecoEsperado` e `DesvioPreco`, porque definem a classe, e o código postal e as coordenadas, porque a localização já está no preço esperado. Também removemos `sqft_above`, que é a área da casa menos a da cave.

In [4]:
vendas["Meses"] = (vendas["date"] - vendas["date"].min()).dt.days / 30.44

NUMERICAS = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "waterfront", "view", "condition", "grade",
             "sqft_basement", "yr_built", "sqft_living15", "sqft_lot15", "Meses"]
ALVO = "ClassePreco"

vendas[NUMERICAS].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
bedrooms,21613.0,3.4,0.9,0.0,3.0,3.0,4.0,11.0
bathrooms,21613.0,2.1,0.8,0.0,1.8,2.2,2.5,8.0
sqft_living,21613.0,2079.9,918.4,290.0,1427.0,1910.0,2550.0,13540.0
sqft_lot,21613.0,15107.0,41420.5,520.0,5040.0,7618.0,10688.0,1651359.0
floors,21613.0,1.5,0.5,1.0,1.0,1.5,2.0,3.5
waterfront,21613.0,0.0,0.1,0.0,0.0,0.0,0.0,1.0
view,21613.0,0.2,0.8,0.0,0.0,0.0,0.0,4.0
condition,21613.0,3.4,0.7,1.0,3.0,3.0,4.0,5.0
grade,21613.0,7.7,1.2,1.0,7.0,7.0,8.0,13.0
sqft_basement,21613.0,291.5,442.6,0.0,0.0,0.0,560.0,4820.0


Separamos 80% das vendas para treino e 20% para teste, com a mesma proporção de cada classe nos dois.

In [5]:
treino, teste = train_test_split(vendas, test_size=0.2, stratify=vendas[ALVO], random_state=SEMENTE)
treino, teste = treino.reset_index(drop=True), teste.reset_index(drop=True)
y_treino, y_teste = treino[ALVO].to_numpy(), teste[ALVO].to_numpy()

print(f"Treino: {len(treino)} | Teste: {len(teste)}")
pd.DataFrame({"treino": treino[ALVO].value_counts(normalize=True),
              "teste": teste[ALVO].value_counts(normalize=True)}).round(3)

Treino: 17290 | Teste: 4323


,treino,teste
ClassePreco,,
normal,0.529,0.529
alto,0.247,0.247
baixo,0.224,0.224


Não há valores em falta e todas as variáveis são números, por isso a preparação só cria duas variáveis a partir de `yr_renovated`. `Renovada` é 1 quando a casa foi renovada e `AnoUltimaObra` é o ano da renovação, ou o da construção se a casa nunca foi renovada.

In [6]:
def preparar(treino, outro):
    """Criar as variáveis da renovação e escolher as colunas que a árvore usa."""
    def transformar(df):
        df = df.copy()
        df["Renovada"] = (df["yr_renovated"] > 0).astype(int)
        # Ano da última obra: o da renovação, ou o da construção se a casa nunca foi renovada
        df["AnoUltimaObra"] = df["yr_renovated"].where(df["yr_renovated"] > 0, df["yr_built"])
        return df[NUMERICAS + ["Renovada", "AnoUltimaObra"]]

    return transformar(treino), transformar(outro)


X_treino, X_teste = preparar(treino, teste)
print(f"{X_treino.shape[1]} variáveis")
X_treino.head()

16 variáveis


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_basement,yr_built,sqft_living15,sqft_lot15,Meses,Renovada,AnoUltimaObra
0,4,2.50,2160,5298,2.5,0,0,4,9,0,1902,1720,2283,0.295664,0,1902
1,4,1.75,1960,9300,1.0,0,0,5,7,620,1979,1500,9752,0.722733,0,1979
2,4,2.50,1950,5602,1.0,0,0,5,7,830,1966,2000,6050,9.428384,0,1966
3,3,1.75,1920,4000,1.0,0,0,3,7,850,1950,2280,4000,4.467806,0,1950
4,3,1.50,2210,6500,1.0,0,0,4,7,1180,1912,1750,6370,7.227332,0,1912


## Treinar a árvore

Estamos a usar o código da árvore em `libs/decisiontrees.py`. Em cada nó escolhe a pergunta `x <= valor` que melhor separa as classes, medida com Gini ou entropia, e para de dividir com `max_depth`, `min_samples_split`, `min_samples_leaf` ou `min_impurity_decrease`. Para a primeira árvore usamos Gini, profundidade máxima 6 para a árvore se conseguir ler, pelo menos 30 vendas por folha para as percentagens de cada folha não virem de poucas vendas, e `class_weight="balanced"`, para garantir que há o mesmo peso nas três classes, já que cerca de metade das vendas são `normal`.

In [7]:
PARAMS_INICIAIS = dict(criterion="gini", max_depth=6, min_samples_leaf=30, class_weight="balanced")

arvore = DecisionTreeClassifier(**PARAMS_INICIAIS).fit(X_treino, y_treino)
print(f"Profundidade: {arvore.get_depth()} | folhas: {arvore.get_n_leaves()}\n")
print(arvore.export_text(max_depth=3))

Profundidade: 6 | folhas: 50

view <= 0.5
|   sqft_basement <= 525.0
|   |   grade <= 7.5
|   |   |   → baixo  (n=6668; alto: 22%, baixo: 25%, normal: 54%)  [...]
|   |   grade > 7.5
|   |   |   → alto  (n=5337; alto: 28%, baixo: 12%, normal: 60%)  [...]
|   sqft_basement > 525.0
|   |   sqft_basement <= 937.5
|   |   |   → baixo  (n=2267; alto: 14%, baixo: 31%, normal: 55%)  [...]
|   |   sqft_basement > 937.5
|   |   |   → baixo  (n=1342; alto: 9%, baixo: 50%, normal: 41%)  [...]
view > 0.5
|   view <= 3.5
|   |   sqft_basement <= 675.0
|   |   |   → alto  (n=743; alto: 54%, baixo: 8%, normal: 38%)  [...]
|   |   sqft_basement > 675.0
|   |   |   → alto  (n=683; alto: 40%, baixo: 22%, normal: 37%)  [...]
|   view > 3.5
|   |   waterfront = 0
|   |   |   → alto  (n=141; alto: 70%, baixo: 4%, normal: 26%)  [...]
|   |   waterfront = 1
|   |   |   → alto  (n=109; alto: 94%, baixo: 2%, normal: 4%)  [...]


In [8]:
# Comparar com a árvore do scikit-learn com os mesmos parâmetros
from sklearn.tree import DecisionTreeClassifier as ArvoreSklearn

referencia = ArvoreSklearn(**PARAMS_INICIAIS, random_state=SEMENTE).fit(X_treino, y_treino)
iguais = (referencia.predict(X_treino) == arvore.predict(X_treino)).mean()
print(f"Previsões iguais às do scikit-learn: {iguais:.1%}")

Previsões iguais às do scikit-learn: 100.0%


## Avaliar

Validação cruzada com 5 folds no treino. Cerca de metade das vendas são `normal`, por isso usamos o F1 macro em vez da accuracy. Comparamos prever sempre `normal`, uma árvore só com as variáveis da regressão e uma árvore com todas.

In [9]:
VARIAVEIS_REGRESSAO = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "Meses"]


def validacao_cruzada(params, colunas=None, folds=5):
    """Prever cada venda do treino com a árvore treinada nos outros folds."""
    previsoes = np.empty(len(treino), dtype=object)
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEMENTE)
    for idx_tr, idx_val in cv.split(treino, y_treino):
        X_tr, X_val = preparar(treino.iloc[idx_tr], treino.iloc[idx_val])
        if colunas is not None:
            X_tr, X_val = X_tr[colunas], X_val[colunas]
        modelo = DecisionTreeClassifier(**params).fit(X_tr, y_treino[idx_tr])
        previsoes[idx_val] = modelo.predict(X_val)
    return previsoes


def metricas(y, previsto):
    recalls = recall_score(y, previsto, labels=CLASSES, average=None, zero_division=0)
    return {"accuracy": accuracy_score(y, previsto),
            "f1_macro": f1_score(y, previsto, labels=CLASSES, average="macro", zero_division=0),
            **{f"recall_{c}": r for c, r in zip(CLASSES, recalls)}}


prev_regressao = validacao_cruzada(PARAMS_INICIAIS, VARIAVEIS_REGRESSAO)
prev_todas = validacao_cruzada(PARAMS_INICIAIS)

pd.DataFrame({
    "Sempre 'normal'": metricas(y_treino, np.full(len(y_treino), "normal")),
    "Árvore: variáveis da regressão": metricas(y_treino, prev_regressao),
    "Árvore: todas as variáveis": metricas(y_treino, prev_todas),
}).T.round(3)

,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Sempre 'normal',0.529,0.231,0.000,1.000,0.000
Árvore: variáveis da regressão,0.410,0.386,0.425,0.444,0.324
Árvore: todas as variáveis,0.490,0.476,0.606,0.478,0.410


Em F1 macro, prever sempre `normal` fica em 0,23. A árvore só com as variáveis da regressão chega a 0,39, por isso a regressão ainda não apanha todo o efeito dessas variáveis. Com as restantes variáveis sobe para 0,48. A accuracy fica abaixo de 53% porque os pesos por classe fazem a árvore prever mais vezes `baixo` e `alto`.

In [10]:
pd.DataFrame(confusion_matrix(y_treino, prev_todas, labels=CLASSES),
             index=[f"real {c}" for c in CLASSES],
             columns=[f"previsto {c}" for c in CLASSES])

,previsto baixo,previsto normal,previsto alto
real baixo,2351,1022,504
real normal,3094,4374,1677
real alto,1058,1460,1750


Só 13% das vendas `baixo` são previstas como `alto`, mas 25% das `alto` são previstas como `baixo`. Mais de metade dos erros são vendas `normal` previstas como `baixo` ou `alto`.

In [11]:
arvore.feature_importances().head(12).round(3)

sqft_basement    0.268
view             0.220
grade            0.150
sqft_living      0.116
yr_built         0.102
condition        0.058
AnoUltimaObra    0.046
sqft_lot         0.017
sqft_living15    0.016
sqft_lot15       0.002
waterfront       0.002
Meses            0.001
dtype: float64

In [12]:
arvore.rules().head(10)

,regra,classe,n,pureza,%alto,%baixo,%normal
0,view <= 0.5 E sqft_basement <= 525.0 E grade > 7.5 E AnoUltimaObra <= 2011.5 E yr_built > 1948.5 E sqft_living <= 3195.0,normal,3792,0.630,0.256,0.114,0.630
1,view <= 0.5 E sqft_basement <= 525.0 E grade <= 7.5 E yr_built > 1967.5 E sqft_living <= 2575.0 E sqft_lot > 3541.5,normal,2093,0.688,0.151,0.161,0.688
2,view <= 0.5 E sqft_basement <= 525.0 E grade <= 7.5 E yr_built <= 1967.5 E condition <= 3.5 E sqft_living <= 1715.0,baixo,1904,0.322,0.252,0.322,0.426
3,view <= 0.5 E sqft_basement <= 525.0 E grade <= 7.5 E yr_built <= 1967.5 E condition > 3.5 E sqft_living <= 1665.0,alto,1484,0.346,0.346,0.167,0.487
4,view <= 0.5 E sqft_basement <= 525.0 E grade > 7.5 E AnoUltimaObra <= 2011.5 E yr_built > 1948.5 E sqft_living > 3195.0,baixo,675,0.246,0.199,0.246,0.556
5,view <= 0.5 E 525.0 < sqft_basement <= 937.5 E 6.5 < grade <= 7.5 E condition > 3.5,normal,623,0.645,0.096,0.258,0.645
6,view <= 0.5 E 525.0 < sqft_basement <= 937.5 E 7.5 < grade <= 8.5 E sqft_living <= 3610.0,baixo,622,0.272,0.170,0.272,0.558
7,view <= 0.5 E 525.0 < sqft_basement <= 937.5 E 6.5 < grade <= 7.5 E condition <= 3.5,baixo,612,0.389,0.087,0.389,0.525
8,view <= 0.5 E sqft_basement > 1125.0 E AnoUltimaObra <= 2008.5 E condition <= 4.5,baixo,540,0.622,0.056,0.622,0.322
9,view <= 0.5 E 937.5 < sqft_basement <= 1125.0 E AnoUltimaObra <= 2008.5 E condition <= 4.5,baixo,535,0.484,0.069,0.484,0.447


In [13]:
# Folhas que preveem baixo ou alto, das mais puras para as menos puras
regras = arvore.rules()
fora_do_normal = regras[(regras["classe"] != "normal") & (regras["n"] >= 30)]
fora_do_normal.sort_values("pureza", ascending=False).head(8)

,regra,classe,n,pureza,%alto,%baixo,%normal
29,view > 3.5 E waterfront = 1 E sqft_living > 2605.0,alto,67,1.000,1.000,0.000,0.000
35,view > 3.5 E waterfront = 0 E sqft_basement <= 1405.0 E Meses > 2.7 E sqft_lot <= 10886.5,alto,40,0.925,0.925,0.000,0.075
39,0.5 < view <= 3.5 E sqft_basement > 675.0 E sqft_living15 > 2035.0 E yr_built <= 1924.5,alto,35,0.886,0.886,0.029,0.086
34,view > 3.5 E waterfront = 1 E sqft_living <= 2605.0,alto,42,0.857,0.857,0.048,0.095
25,view <= 0.5 E sqft_basement <= 525.0 E grade > 8.5 E AnoUltimaObra > 2011.5 E sqft_living <= 2775.0,alto,99,0.768,0.768,0.020,0.212
41,view <= 0.5 E sqft_basement <= 525.0 E grade <= 7.5 E yr_built > 1967.5 E sqft_living > 3125.0,baixo,35,0.743,0.057,0.743,0.200
21,0.5 < view <= 3.5 E sqft_basement <= 675.0 E yr_built <= 1971.5 E grade > 7.5 E sqft_lot15 <= 14403.0,alto,166,0.735,0.735,0.018,0.247
38,view > 3.5 E waterfront = 0 E sqft_basement <= 1405.0 E Meses > 2.7 E sqft_lot > 10886.5,alto,37,0.730,0.730,0.027,0.243


A área da cave é a variável mais importante, seguida da vista e da qualidade da construção. A regressão usa a área total da casa, que inclui a cave, e as casas com caves grandes ficam abaixo do esperado, provavelmente porque a área da cave vale menos do que a área acima do chão. As casas com a melhor vista, à beira de água e com mais de 2605 sqft são todas `alto`. As casas com vista de 1 a 3, mais de 675 sqft de cave, vizinhos com casas de mais de 2035 sqft e construídas até 1924 têm 89% de vendas `alto`. No lado oposto, as casas sem vista, com mais de 1125 sqft de cave, em estado até 4 e sem obras desde 2008 têm 62% de vendas `baixo`. O mês da venda quase não conta, porque já está na regressão.

## Otimizar os hiperparâmetros

Testamos todas as combinações de `criterion`, `class_weight`, `max_depth`, `min_samples_leaf` e `min_samples_split` com a mesma validação cruzada. Em `class_weight` comparamos `balanced` com a raiz quadrada desses pesos, que fica a meio caminho entre não usar pesos e `balanced`. Para cada combinação guardamos o F1 macro médio no treino e na validação. Os folds são preparados antes da pesquisa e reutilizados em todas as combinações.

In [14]:
from itertools import product
from time import perf_counter

from joblib import Parallel, delayed

# Pesos balanced: n / (3 * n_classe). Os pesos raiz são a raiz quadrada destes
frequencia = pd.Series(y_treino).value_counts(normalize=True)
PESOS = {"balanced": "balanced", "raiz": (1 / (3 * frequencia) ** 0.5).round(3).to_dict()}
print("Pesos raiz:", PESOS["raiz"])

GRELHA = {
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced", "raiz"],
    "max_depth": [6, 8, 10, 12],
    "min_samples_leaf": [10, 20, 40, 80],
    "min_samples_split": [2, 150],
}

# Preparar os 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
folds = []
for idx_tr, idx_val in cv.split(treino, y_treino):
    X_tr, X_val = preparar(treino.iloc[idx_tr], treino.iloc[idx_val])
    folds.append((X_tr, y_treino[idx_tr], X_val, y_treino[idx_val]))

def avaliar(combinacao):
    """F1 macro médio no treino e na validação de uma combinação, nos 5 folds."""
    params = {**combinacao, "class_weight": PESOS[combinacao["class_weight"]]}
    f1_tr, f1_val = [], []
    for X_tr, y_tr, X_val, y_val in folds:
        modelo = DecisionTreeClassifier(**params).fit(X_tr, y_tr)
        f1_tr.append(f1_score(y_tr, modelo.predict(X_tr), average="macro"))
        f1_val.append(f1_score(y_val, modelo.predict(X_val), average="macro"))
    return {**combinacao, "f1_treino": np.mean(f1_tr), "f1_validacao": np.mean(f1_val), "desvio_validacao": np.std(f1_val)}


# Avaliar as combinações em paralelo, uma por núcleo do processador
inicio = perf_counter()
combinacoes = [dict(zip(GRELHA, valores)) for valores in product(*GRELHA.values())]
resultados = Parallel(n_jobs=-1)(delayed(avaliar)(c) for c in combinacoes)

resultados = pd.DataFrame(resultados).sort_values("f1_validacao", ascending=False, ignore_index=True)
print(f"{len(resultados)} combinações em {perf_counter() - inicio:.0f} s")
resultados.head(10).round(3)

Pesos raiz: {'normal': 0.794, 'alto': 1.162, 'baixo': 1.219}
128 combinações em 119 s


,criterion,class_weight,max_depth,min_samples_leaf,min_samples_split,f1_treino,f1_validacao,desvio_validacao
0,gini,raiz,8,10,150,0.535,0.501,0.005
1,entropy,raiz,12,10,150,0.556,0.500,0.009
2,gini,raiz,8,40,2,0.539,0.500,0.008
3,entropy,raiz,10,20,150,0.545,0.500,0.005
4,entropy,raiz,10,10,150,0.545,0.500,0.007
5,gini,raiz,8,20,150,0.535,0.500,0.005
6,entropy,raiz,12,20,150,0.553,0.499,0.006
7,entropy,raiz,10,40,2,0.553,0.499,0.007
8,entropy,raiz,10,40,150,0.544,0.499,0.007
9,gini,raiz,8,40,150,0.533,0.498,0.007


Melhor combinação para cada profundidade, com o F1 no treino ao lado do F1 na validação.

In [15]:
por_profundidade = resultados.loc[resultados.groupby("max_depth")["f1_validacao"].idxmax()]
por_profundidade[["max_depth", "criterion", "class_weight", "min_samples_leaf", "min_samples_split", "f1_treino", "f1_validacao", "desvio_validacao"]].round(3)

,max_depth,criterion,class_weight,min_samples_leaf,min_samples_split,f1_treino,f1_validacao,desvio_validacao
65,6,gini,raiz,10,150,0.504,0.485,0.006
0,8,gini,raiz,10,150,0.535,0.501,0.005
3,10,entropy,raiz,20,150,0.545,0.500,0.005
1,12,entropy,raiz,10,150,0.556,0.500,0.009


In [16]:
# Efeito médio de cada hiperparâmetro no F1 de validação
for nome in GRELHA:
    print(resultados.groupby(nome)["f1_validacao"].mean().round(3).to_dict())

{'entropy': 0.481, 'gini': 0.487}
{'balanced': 0.477, 'raiz': 0.492}
{6: 0.469, 8: 0.492, 10: 0.491, 12: 0.486}
{10: 0.484, 20: 0.485, 40: 0.486, 80: 0.483}
{2: 0.483, 150: 0.485}


O peso das classes é o hiperparâmetro que mais conta. Os pesos raiz ficam em média 1,5 pontos acima de `balanced` e estão em todas as dez melhores combinações. A profundidade também conta, as árvores com 8 a 10 níveis ficam cerca de 2 pontos acima das de 6 e as de 12 já descem um pouco. Gini e entropia dão quase o mesmo, tal como `min_samples_leaf` e `min_samples_split`. Nas melhores combinações a diferença entre treino e validação fica abaixo de 6 pontos.

In [17]:
MELHORES = resultados.iloc[0][list(GRELHA)].to_dict()
MELHORES["class_weight"] = PESOS[MELHORES["class_weight"]]
for nome in ["max_depth", "min_samples_leaf", "min_samples_split"]:
    MELHORES[nome] = int(MELHORES[nome])
print(MELHORES)

prev_melhor = validacao_cruzada(MELHORES)
pd.DataFrame({
    "Árvore inicial": metricas(y_treino, prev_todas),
    "Árvore otimizada": metricas(y_treino, prev_melhor),
}).T.round(3)

{'criterion': 'gini', 'class_weight': {'normal': 0.794, 'alto': 1.162, 'baixo': 1.219}, 'max_depth': 8, 'min_samples_leaf': 10, 'min_samples_split': 150}


,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Árvore inicial,0.490,0.476,0.606,0.478,0.410
Árvore otimizada,0.539,0.501,0.446,0.633,0.422


A árvore escolhida usa Gini, os pesos raiz, `max_depth=8`, `min_samples_leaf=10` e `min_samples_split=150`. Em relação à árvore inicial o F1 macro sobe de 0,48 para 0,50 e a accuracy de 0,49 para 0,54. Com pesos mais baixos nas classes `baixo` e `alto`, a árvore acerta mais vendas `normal`, 63% contra 48%, e menos vendas `baixo`, 45% contra 61%.

## Analisar o modelo final

A árvore final é treinada com os melhores hiperparâmetros em todo o conjunto de treino.

In [18]:
final = DecisionTreeClassifier(**MELHORES).fit(X_treino, y_treino)
print(f"Profundidade: {final.get_depth()} | folhas: {final.get_n_leaves()}\n")
print(final.export_text(max_depth=3))

Profundidade: 8 | folhas: 86

view <= 0.5
|   sqft_basement <= 525.0
|   |   yr_built <= 1964.5
|   |   |   → normal  (n=4295; alto: 30%, baixo: 26%, normal: 45%)  [...]
|   |   yr_built > 1964.5
|   |   |   → normal  (n=7710; alto: 22%, baixo: 15%, normal: 63%)  [...]
|   sqft_basement > 525.0
|   |   sqft_basement <= 937.5
|   |   |   → normal  (n=2267; alto: 14%, baixo: 31%, normal: 55%)  [...]
|   |   sqft_basement > 937.5
|   |   |   → baixo  (n=1342; alto: 9%, baixo: 50%, normal: 41%)  [...]
view > 0.5
|   view <= 3.5
|   |   sqft_basement <= 675.0
|   |   |   → alto  (n=743; alto: 54%, baixo: 8%, normal: 38%)  [...]
|   |   sqft_basement > 675.0
|   |   |   → alto  (n=683; alto: 40%, baixo: 22%, normal: 37%)  [...]
|   view > 3.5
|   |   waterfront = 0
|   |   |   → alto  (n=141; alto: 70%, baixo: 4%, normal: 26%)
|   |   waterfront = 1
|   |   |   → alto  (n=109; alto: 94%, baixo: 2%, normal: 4%)


In [19]:
final.feature_importances().head(12).round(3)

sqft_basement    0.205
view             0.186
sqft_living      0.148
grade            0.132
yr_built         0.111
AnoUltimaObra    0.060
condition        0.057
sqft_living15    0.028
sqft_lot         0.022
sqft_lot15       0.017
bathrooms        0.015
bedrooms         0.008
dtype: float64

In [20]:
# Folhas com pelo menos 30 vendas onde a árvore prevê baixo ou alto, das mais puras para as menos puras
regras_finais = final.rules()
regras_finais[(regras_finais["classe"] != "normal") & (regras_finais["n"] >= 30)].sort_values("pureza", ascending=False).head(10)

,regra,classe,n,pureza,%alto,%baixo,%normal
37,view > 3.5 E waterfront = 1,alto,109,0.945,0.945,0.018,0.037
68,0.5 < view <= 3.5 E sqft_basement <= 675.0 E yr_built <= 1971.5 E grade > 7.5 E 8209.5 < sqft_lot15 <= 14403.0,alto,50,0.860,0.860,0.020,0.120
50,view <= 0.5 E sqft_basement <= 525.0 E yr_built > 1964.5 E grade > 8.5 E AnoUltimaObra > 2012.5 E sqft_living <= 2775.0,alto,80,0.775,0.775,0.025,0.200
73,view <= 0.5 E sqft_basement <= 525.0 E yr_built > 1964.5 E 7.5 < grade <= 8.5 E AnoUltimaObra > 2012.5 E sqft_living <= 2775.0 E sqft_lot15 <= 1416.5,alto,34,0.706,0.706,0.000,0.294
25,view > 3.5 E waterfront = 0,alto,141,0.695,0.695,0.043,0.262
31,0.5 < view <= 3.5 E sqft_basement <= 675.0 E yr_built <= 1971.5 E grade > 7.5 E sqft_lot15 <= 8209.5,alto,116,0.681,0.681,0.017,0.302
47,0.5 < view <= 3.5 E 675.0 < sqft_basement <= 1175.0 E yr_built > 1938.5 E sqft_living15 > 2550.0 E sqft_living <= 3305.0,alto,88,0.670,0.670,0.091,0.239
9,view <= 0.5 E sqft_basement > 1125.0 E AnoUltimaObra <= 2006.5 E condition <= 4.5 E sqft_living15 <= 3075.0,baixo,440,0.666,0.045,0.666,0.289
23,view <= 0.5 E 937.5 < sqft_basement <= 1125.0 E AnoUltimaObra <= 1982.5 E condition <= 4.5 E sqft_living15 <= 1745.0,baixo,147,0.653,0.034,0.653,0.313
42,1.5 < view <= 3.5 E sqft_basement <= 675.0 E yr_built > 1971.5 E sqft_living15 > 2215.0 E sqft_lot15 > 4593.5 E sqft_living <= 3205.0,alto,101,0.644,0.644,0.020,0.337


In [21]:
# F1 macro pela qualidade da construção, com as previsões da validação cruzada da árvore otimizada
grupo_grade = pd.cut(treino["grade"], [0, 6, 7, 8, 13], labels=["até 6", "7", "8", "9 ou mais"])
pd.DataFrame({
    g: {"n": int((grupo_grade == g).sum()),
        **metricas(y_treino[grupo_grade == g], prev_melhor[grupo_grade == g])}
    for g in grupo_grade.cat.categories
}).T.round(3)

,n,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
até 6,1862.0,0.418,0.374,0.769,0.154,0.323
7,7191.0,0.534,0.473,0.410,0.654,0.347
8,4840.0,0.579,0.485,0.296,0.772,0.351
9 ou mais,3397.0,0.560,0.524,0.405,0.569,0.606


A árvore final usa mais a cave, a vista, a área da casa, a qualidade da construção e o ano de construção. As casas com a melhor vista e à beira de água têm 95% de vendas `alto`, e as que têm a melhor vista sem estar à beira de água têm 70%. As casas sem vista com `grade` acima de 8, construídas ou renovadas depois de 2012 e com até 2775 sqft têm 78% de vendas `alto`. No lado oposto, as casas sem vista, com mais de 1125 sqft de cave, em estado até 4 e sem obras desde 2006 têm 67% de vendas `baixo`, três vezes a média. Os quartos e as casas de banho praticamente não contam, porque já estão na regressão.

O F1 macro sobe com a qualidade da construção, de 0,37 nas casas com `grade` até 6 para 0,52 nas casas com 9 ou mais. Nas casas com `grade` até 6 a árvore encontra 77% das vendas `baixo` mas só 15% das `normal`.

## Random forest

Uma árvore sozinha muda bastante com pequenas mudanças no treino, porque uma divisão diferente no topo muda tudo o que fica por baixo. A random forest treina muitas árvores e faz a média das probabilidades que cada uma dá. Cada árvore é treinada numa amostra das vendas do treino tirada com reposição, do mesmo tamanho do treino, e em cada nó só pode escolher a divisão entre algumas variáveis sorteadas, o `max_features`. Assim as árvores ficam diferentes umas das outras e os erros de cada uma compensam-se em parte. O código está em `libs/randomforest.py` e usa a árvore de `libs/decisiontrees.py`.

Na forest juntamos às variáveis da árvore as cinco variáveis criadas no `01_EDA`, que são a parte da casa que é cave, a área da casa e do terreno face às 15 casas vizinhas, os anos desde a última obra e a área por quarto. Usamos os pesos raiz e testamos `max_features` e `min_samples_leaf` com 50 árvores, treinadas em paralelo, e os mesmos folds da árvore.

In [22]:
from libs.randomforest import RandomForestClassifier

NOVAS = ["FracaoCave", "AreaVsVizinhos", "LoteVsVizinhos", "AnosSemObras", "AreaPorQuarto"]


def preparar_forest(treino, outro):
    """As variáveis da árvore mais as cinco variáveis novas."""
    X_treino, X_outro = preparar(treino, outro)
    return pd.concat([X_treino, treino[NOVAS]], axis=1), pd.concat([X_outro, outro[NOVAS]], axis=1)


def validacao_cruzada_forest(params):
    """Prever cada venda do treino com a forest treinada nos outros folds."""
    previsoes = np.empty(len(treino), dtype=object)
    for X_tr, X_val, idx_tr, idx_val in folds_forest:
        modelo = RandomForestClassifier(**params).fit(X_tr, y_treino[idx_tr])
        previsoes[idx_val] = modelo.predict(X_val)
    return previsoes


folds_forest = [(*preparar_forest(treino.iloc[idx_tr], treino.iloc[idx_val]), idx_tr, idx_val)
                for idx_tr, idx_val in cv.split(treino, y_treino)]

GRELHA_FOREST = {"max_features": ["sqrt", 0.5], "min_samples_leaf": [20, 40]}

resultados_forest = []
inicio = perf_counter()
for valores in product(*GRELHA_FOREST.values()):
    combinacao = dict(zip(GRELHA_FOREST, valores))
    params = {**combinacao, "n_estimators": 50, "class_weight": PESOS["raiz"],
              "random_state": SEMENTE, "n_jobs": -1}
    resultados_forest.append({**combinacao, **metricas(y_treino, validacao_cruzada_forest(params))})

print(f"{len(resultados_forest)} combinações em {perf_counter() - inicio:.0f} s")
resultados_forest = pd.DataFrame(resultados_forest).sort_values("f1_macro", ascending=False, ignore_index=True)
resultados_forest.round(3)

4 combinações em 180 s


,max_features,min_samples_leaf,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
0,0.5,20,0.573,0.528,0.432,0.698,0.433
1,0.5,40,0.567,0.525,0.436,0.681,0.442
2,sqrt,20,0.571,0.524,0.414,0.700,0.437
3,sqrt,40,0.567,0.522,0.415,0.692,0.439


TEXTO_GRELHA

In [23]:
PARAMS_FOREST = {**resultados_forest.iloc[0][list(GRELHA_FOREST)].to_dict(),
                 "n_estimators": 100, "class_weight": PESOS["raiz"], "random_state": SEMENTE, "n_jobs": -1}
PARAMS_FOREST["min_samples_leaf"] = int(PARAMS_FOREST["min_samples_leaf"])
print(PARAMS_FOREST)

# Validação cruzada da forest com 100 árvores, ao lado da árvore otimizada
prev_forest = validacao_cruzada_forest(PARAMS_FOREST)
pd.DataFrame({
    "Árvore otimizada": metricas(y_treino, prev_melhor),
    "Random forest": metricas(y_treino, prev_forest),
}).T.round(3)

{'max_features': 0.5, 'min_samples_leaf': 20, 'n_estimators': 100, 'class_weight': {'normal': 0.794, 'alto': 1.162, 'baixo': 1.219}, 'random_state': 42, 'n_jobs': -1}


,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Árvore otimizada,0.539,0.501,0.446,0.633,0.422
Random forest,0.573,0.529,0.432,0.699,0.433


In [24]:
# Forest final, treinada em todo o conjunto de treino
X_treino_forest, X_teste_forest = preparar_forest(treino, teste)
forest = RandomForestClassifier(**PARAMS_FOREST).fit(X_treino_forest, y_treino)
forest.feature_importances().head(12).round(3)

AreaVsVizinhos    0.115
AreaPorQuarto     0.075
view              0.069
sqft_living       0.069
sqft_lot15        0.065
yr_built          0.061
sqft_lot          0.061
grade             0.059
sqft_living15     0.057
Meses             0.055
AnosSemObras      0.055
sqft_basement     0.053
dtype: float64

TEXTO_FOREST

## Avaliar no teste

In [25]:
prev_teste = final.predict(X_teste)
prev_teste_forest = forest.predict(X_teste_forest)
pd.DataFrame({
    "Sempre 'normal'": metricas(y_teste, np.full(len(y_teste), "normal")),
    "Árvore final": metricas(y_teste, prev_teste),
    "Random forest": metricas(y_teste, prev_teste_forest),
}).T.round(3)

,accuracy,f1_macro,recall_baixo,recall_normal,recall_alto
Sempre 'normal',0.529,0.231,0.000,1.000,0.000
Árvore final,0.560,0.515,0.434,0.684,0.409
Random forest,0.592,0.553,0.475,0.704,0.457


In [26]:
pd.DataFrame(confusion_matrix(y_teste, prev_teste, labels=CLASSES),
             index=[f"real {c}" for c in CLASSES],
             columns=[f"previsto {c}" for c in CLASSES])

,previsto baixo,previsto normal,previsto alto
real baixo,421,454,95
real normal,359,1564,363
real alto,115,516,436


In [27]:
# Matriz de confusão da random forest
pd.DataFrame(confusion_matrix(y_teste, prev_teste_forest, labels=CLASSES),
             index=[f"real {c}" for c in CLASSES],
             columns=[f"previsto {c}" for c in CLASSES])

,previsto baixo,previsto normal,previsto alto
real baixo,461,438,71
real normal,323,1610,353
real alto,99,480,488


No teste a árvore tem F1 macro de 0,52, um pouco acima dos 0,50 da validação cruzada. Fica acima de prever sempre `normal`, que tem 0,23, e também em accuracy, 0,56 contra 0,53. A árvore confunde pouco as classes extremas, só 10% das vendas `baixo` são previstas como `alto` e 11% das `alto` como `baixo`. Pouco mais de metade dos erros são vendas `baixo` e `alto` previstas como `normal`.